Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas dipindahkan ke PySpark, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, membaca data langsung dari HDFS.

Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS.
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS.
!hdfs dfs -mkdir -p /user/caitlyn/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/caitlyn/tugas4/
print("Berhasil diunggah ke HDFS: /user/caitlyn/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/caitlyn/tugas4/transaksi_september_2026.csv


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, sum as spark_sum, avg

# Membuat SparkSession baru.
spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 05:33:10 WARN Utils: Your hostname, caitlyn-ThinkPad-E570, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlp5s0)
26/09/16 05:33:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 05:33:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


A. Membaca dan Eksplorasi Awal (bobot 15%)

Baca dataset dari HDFS, tampilkan printSchema(), jumlah baris (count()), dan 10 baris pertama (show(10)).

In [3]:
# Membaca data langsung dari HDFS.
df_tugas = spark.read.csv("hdfs://localhost:9000/user/caitlyn/tugas4/transaksi_september_2026.csv", header=True, inferSchema=True)

df_tugas.printSchema()
print("Jumlah baris:", df_tugas.count())
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

B. Menangani Data Kosong (bobot 15%)

Kolom rating memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan df.na.fill() atau df.na.drop() (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [4]:
# Menghitung data kosong
jumlah_kosong = df_tugas.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_kosong}")

# Menangani data kosong dengan membuangnya (drop)
df_tugas_bersih = df_tugas.na.drop(subset=["rating"])
print("Jumlah baris setelah dibersihkan:", df_tugas_bersih.count())

Jumlah baris dengan rating kosong: 204
Jumlah baris setelah dibersihkan: 796


Alasan penggunaan na.drop(): Dalam analisis rating produk, mengisi nilai yang kosong (misalnya dengan angka 0 atau nilai rata-rata) berisiko merusak validitas sentimen pelanggan yang asli. Lebih baik membuang baris yang tidak memiliki rating tersebut agar perhitungan rata-rata rating nantinya tetap murni berdasarkan pelanggan yang benar-benar memberikan ulasan.

C. Transformasi Data (bobot 20%)

Tambahkan kolom total_pendapatan (unit_terjual x harga_satuan), lalu tambahkan kolom tier_transaksi yang bernilai "Besar" jika total_pendapatan > 500000, atau "Kecil" jika sebaliknya 

In [5]:
# Tambah kolom total_pendapatan.
df_tugas_bersih = df_tugas_bersih.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Tambah kolom tier_transaksi menggunakan fungsi when()
df_tugas_bersih = df_tugas_bersih.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

# Menampilkan hasil transformasi.
df_tugas_bersih.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows


D. Analisis dengan GroupBy (bobot 30%)

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki total_pendapatan tertinggi?
2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
3. Berapa rata-rata rating untuk masing-masing metode_pembayaran (data kosong sudah ditangani di bagian B)?

In [6]:
# 1. Kategori dengan total_pendapatan tertinggi.
print("1. Kategori Pendapatan Tertinggi:")
df_tugas_bersih.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total")).orderBy(col("total").desc()).show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak.
print("2. Kota dengan Transaksi Besar Terbanyak:")
df_tugas_bersih.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota").count().orderBy(col("count").desc()).show(1)

# 3. Rata-rata rating per metode pembayaran.
print("3. Rata-rata Rating per Metode Pembayaran:")
df_tugas_bersih.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).show()

1. Kategori Pendapatan Tertinggi:
+------------+---------+
|    kategori|    total|
+------------+---------+
|Rumah Tangga|108285000|
+------------+---------+
only showing top 1 row
2. Kota dengan Transaksi Besar Terbanyak:
+----+-----+
|kota|count|
+----+-----+
|Solo|   74|
+----+-----+
only showing top 1 row
3. Rata-rata Rating per Metode Pembayaran:
+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|     Kartu Kredit|4.109947643979058|
|         E-Wallet|4.135678391959799|
+-----------------+-----------------+



E. Menyimpan Hasil ke HDFS (bobot 20%)

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom total_pendapatan dan tier_transaksi) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

Catatan: Spark menyimpan hasil sebagai beberapa berkas partisi (part-00000..., dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark.

Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi!

In [7]:
# Menyimpan kembali DataFrame yang sudah bersih dan ditransformasi ke HDFS.
df_tugas_bersih.write.mode("overwrite").csv("hdfs://localhost:9000/user/caitlyn/tugas4/hasil_olahan.csv", header=True)
print("Berhasil disimpan ke HDFS!")

Berhasil disimpan ke HDFS!


Penjelasan Partisi: Spark menyimpan hasil ekspor ke dalam beberapa berkas partisi (seperti part-00000...) karena sifat arsitekturnya yang terdistribusi. Setiap executor di Spark menulis bagian datanya masing-masing secara paralel untuk mempercepat proses penyimpanan. Jika data dipaksa digabung menjadi satu berkas tunggal, keuntungan kecepatan pemrosesan paralel dari Spark akan hilang.